# Model Evaluation — All Pipelines

Evaluates all four trained pipelines against the **2024–2025 test set** (held-out tournament seasons).

| Pipeline | Notebook | Description |
|---|---|---|
| **With Seeds** | `2_train_2026.ipynb` / `3_scores_2026.ipynb` | XGBClassifier + regressors, includes seed features |
| **No Seeds** | `4_no_seed_2026.ipynb` | Same architecture, seed features removed |
| **Kaggle** | `kaggle_model_2026.ipynb` | LOSO XGBoost + spline calibration, ELO + GLM features |
| **Per-Round** | `bracket_rounds_local_2026.ipynb` | Separate model trained per tournament round |

**Test set:** Seasons 2024 and 2025 (63 games each = 126 total men's tournament games, augmented to 252 with W/L swap)

In [2]:
import warnings

import joblib
import numpy as np
import pandas as pd
import statsmodels.api as sm
import tqdm
import xgboost as xgb
from sklearn.metrics import accuracy_score, brier_score_loss, mean_absolute_error
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings('ignore')

TEST_SEASONS = [2024, 2025]

---
## 1. With Seeds & No Seeds Pipelines
Uses `data_2026/final_features.csv` — pre-built feature matrix from `1_cleaning_2026.ipynb`.

In [3]:
final = (
    pd.read_csv('data_2026/final_features.csv')
    .rename(columns={
        'WTEAMID': 'W_TEAMID', 'LTEAMID': 'L_TEAMID',
        'WSCORE': 'W_SCORE',   'LSCORE': 'L_SCORE',
    })
)
final['WIN_INDICATOR'] = 1

drop_cols = ['W_CTWINS', 'W_AVERAGECTSCORE', 'L_CTWINS', 'L_AVERAGECTSCORE',
             'W_WLOCN', 'W_WLOCH', 'W_WLOCA', 'L_WLOCN', 'L_WLOCH', 'L_WLOCA']
final = final.drop(columns=[c for c in drop_cols if c in final.columns])

NON_FEATURE_COLS = {'SEASON', 'WIN_INDICATOR', 'L_TEAMID', 'W_TEAMID',
                    'W_SCORE', 'L_SCORE', 'ROUND', 'L_REGION', 'W_REGION'}
feature_cols    = [c for c in final.columns if c not in NON_FEATURE_COLS]
no_seed_cols    = [c for c in feature_cols if 'SEED' not in c.upper()]

# Build augmented test set
def swap_wl(df):
    out = df.copy()
    w_cols = sorted([c for c in df.columns if c.startswith('W_')])
    l_cols = ['L_' + c[2:] for c in w_cols]
    for w, l in zip(w_cols, l_cols):
        out[w] = df[l]
        out[l] = df[w]
    return out

test_raw = final[final.SEASON.isin(TEST_SEASONS)].copy()
test_aug = pd.concat([test_raw, swap_wl(test_raw)], ignore_index=True)
test_aug['WIN_INDICATOR']  = (test_aug['W_SCORE'] > test_aug['L_SCORE']).astype(int)
test_aug['ACTUAL_SPREAD'] = test_aug['W_SCORE'] - test_aug['L_SCORE']
test_aug['ACTUAL_TOTAL']  = test_aug['W_SCORE'] + test_aug['L_SCORE']

print(f'feature_cols: {len(feature_cols)},  no_seed_cols: {len(no_seed_cols)}')
print(f'Test games (augmented): {len(test_aug)} ({len(test_raw)} unique)')

feature_cols: 164,  no_seed_cols: 162
Test games (augmented): 268 (134 unique)


In [4]:
# ── Load models ──────────────────────────────────────────────────────────────
model_seeds    = joblib.load('model/march_madness_model_scores.joblib')   # win, with seeds
model_no_seed  = joblib.load('model/march_madness_model_no_seed.joblib')  # win, no seeds
spread_model   = joblib.load('model/spread_model.joblib')                  # spread, with seeds
total_model    = joblib.load('model/total_model.joblib')                   # total, with seeds
spread_no_seed = joblib.load('model/spread_model_no_seed.joblib')         # spread, no seeds
total_no_seed  = joblib.load('model/total_model_no_seed.joblib')          # total, no seeds
print('Models loaded.')

Models loaded.


In [5]:
def eval_standard(win_m, spread_m, total_m, feat_cols, label):
    X = test_aug[feat_cols]
    win_acc    = accuracy_score(test_aug['WIN_INDICATOR'],  win_m.predict(X))
    spread_mae = mean_absolute_error(test_aug['ACTUAL_SPREAD'], spread_m.predict(X))
    total_mae  = mean_absolute_error(test_aug['ACTUAL_TOTAL'],  total_m.predict(X))
    print(f'{label:30s}  Win: {win_acc*100:.2f}%   Spread MAE: {spread_mae:.2f} pts   Total MAE: {total_mae:.2f} pts')
    return {'Pipeline': label, 'Win Accuracy': round(win_acc*100,2), 'Spread MAE': round(spread_mae,2), 'Total MAE': round(total_mae,2)}

rows = []
rows.append(eval_standard(model_seeds,   spread_model,   total_model,   feature_cols,  'With Seeds'))
rows.append(eval_standard(model_no_seed, spread_no_seed, total_no_seed, no_seed_cols,  'No Seeds'))

# Per-season breakdown
print()
for season in TEST_SEASONS:
    sub = test_aug[test_aug.SEASON == season]
    print(f'  With Seeds  {season}  Win: {accuracy_score(sub.WIN_INDICATOR, model_seeds.predict(sub[feature_cols]))*100:.2f}%')
for season in TEST_SEASONS:
    sub = test_aug[test_aug.SEASON == season]
    print(f'  No Seeds    {season}  Win: {accuracy_score(sub.WIN_INDICATOR, model_no_seed.predict(sub[no_seed_cols]))*100:.2f}%')

With Seeds                      Win: 73.51%   Spread MAE: 10.07 pts   Total MAE: 14.16 pts
No Seeds                        Win: 63.06%   Spread MAE: 11.32 pts   Total MAE: 14.16 pts

  With Seeds  2024  Win: 70.15%
  With Seeds  2025  Win: 76.87%
  No Seeds    2024  Win: 57.46%
  No Seeds    2025  Win: 68.66%


---
## 2. Per-Round Pipeline
Trains round-specific models on 2010–2023, evaluates on 2024–2025.

In [6]:
train_raw = final[(final.SEASON >= 2010) & (final.SEASON <= 2023)].copy()
train_aug = pd.concat([train_raw, swap_wl(train_raw)], ignore_index=True)
train_aug['WIN_INDICATOR'] = (train_aug['W_SCORE'] > train_aug['L_SCORE']).astype(int)

WIN_PARAMS    = dict(learning_rate=0.1, max_depth=4, min_child_weight=4, n_estimators=100)
SPREAD_PARAMS = dict(learning_rate=0.1, max_depth=3, min_child_weight=2, n_estimators=100)
TOTAL_PARAMS  = dict(learning_rate=0.1, max_depth=3, min_child_weight=2, n_estimators=100)

round_names = {1: 'R64', 2: 'R32', 3: 'S16', 4: 'E8', 5: 'FF', 6: 'Champ'}
round_models_eval  = {}
spread_models_eval = {}
total_models_eval  = {}

for r in range(1, 7):
    rt = train_aug[train_aug.ROUND == r]
    if len(rt) == 0:
        continue
    X, y_win = rt[feature_cols], rt['WIN_INDICATOR']
    round_models_eval[r]  = XGBClassifier(eval_metric='logloss', **WIN_PARAMS).fit(X, y_win)
    spread_models_eval[r] = XGBRegressor(eval_metric='rmse', **SPREAD_PARAMS).fit(X, rt['W_SCORE'] - rt['L_SCORE'])
    total_models_eval[r]  = XGBRegressor(eval_metric='rmse', **TOTAL_PARAMS).fit(X,  rt['W_SCORE'] + rt['L_SCORE'])

print(f'Trained {len(round_models_eval)} rounds × 3 model types = {len(round_models_eval)*3} models')

Trained 6 rounds × 3 model types = 18 models


In [7]:
round_rows = []
for r, model in round_models_eval.items():
    tr = test_aug[test_aug.ROUND == r]
    if len(tr) == 0:
        continue
    win_acc    = accuracy_score(tr['WIN_INDICATOR'],  model.predict(tr[feature_cols]))
    spread_mae = mean_absolute_error(tr['ACTUAL_SPREAD'], spread_models_eval[r].predict(tr[feature_cols]))
    total_mae  = mean_absolute_error(tr['ACTUAL_TOTAL'],  total_models_eval[r].predict(tr[feature_cols]))
    round_rows.append({'Round': r, 'Name': round_names[r], 'N_Games': len(tr)//2,
                       'Win_Acc%': round(win_acc*100,2),
                       'Spread_MAE': round(spread_mae,2), 'Total_MAE': round(total_mae,2)})

pr_df = pd.DataFrame(round_rows)
display(pr_df)

# Build a test subset that only includes rounds with trained models
test_with_round = pd.concat([test_aug[test_aug.ROUND == r]
                              for r in round_models_eval
                              if len(test_aug[test_aug.ROUND == r]) > 0],
                             ignore_index=True)

overall_pr_win = accuracy_score(
    test_with_round['WIN_INDICATOR'],
    np.concatenate([round_models_eval[r].predict(test_with_round[test_with_round.ROUND==r][feature_cols])
                    for r in round_models_eval if len(test_with_round[test_with_round.ROUND==r]) > 0])
)
overall_pr_spread = mean_absolute_error(
    test_with_round['ACTUAL_SPREAD'],
    np.concatenate([spread_models_eval[r].predict(test_with_round[test_with_round.ROUND==r][feature_cols])
                    for r in round_models_eval if len(test_with_round[test_with_round.ROUND==r]) > 0])
)
overall_pr_total = mean_absolute_error(
    test_with_round['ACTUAL_TOTAL'],
    np.concatenate([total_models_eval[r].predict(test_with_round[test_with_round.ROUND==r][feature_cols])
                    for r in round_models_eval if len(test_with_round[test_with_round.ROUND==r]) > 0])
)

print(f'\nPer-Round overall  Win: {overall_pr_win*100:.2f}%   Spread MAE: {overall_pr_spread:.2f} pts   Total MAE: {overall_pr_total:.2f} pts')

rows.append({'Pipeline': 'Per-Round', 'Win Accuracy': round(overall_pr_win*100,2),
             'Spread MAE': round(overall_pr_spread,2), 'Total MAE': round(overall_pr_total,2)})

,Round,Name,N_Games,Win_Acc%,Spread_MAE,Total_MAE
0,1,R64,64,77.34,9.78,12.19
1,2,R32,32,75.00,10.33,14.62
2,3,S16,16,59.38,9.47,19.27
3,4,E8,4,75.00,8.99,16.23
4,5,FF,4,25.00,7.00,13.78
5,6,Champ,6,58.33,11.87,15.72



Per-Round overall  Win: 71.83%   Spread MAE: 9.87 pts   Total MAE: 14.05 pts


---
## 3. Kaggle Pipeline

Rebuilds the feature engineering (seeds, season averages, ELO, GLM quality) to evaluate the LOSO models on the 2024–2025 test seasons.  
The win model uses **OOF holdout predictions** (each season's model was trained without that season).  
The spread/total regressors were trained on all historical data — their 2024–2025 metrics are **in-sample** and optimistically biased.

In [8]:
data_dir = 'data_2026'

M_regular = pd.read_csv(f'{data_dir}/MRegularSeasonDetailedResults.csv')
M_tourney  = pd.read_csv(f'{data_dir}/MNCAATourneyDetailedResults.csv')
M_seeds    = pd.read_csv(f'{data_dir}/MNCAATourneySeeds.csv')
W_regular  = pd.read_csv(f'{data_dir}/WRegularSeasonDetailedResults.csv')
W_tourney  = pd.read_csv(f'{data_dir}/WNCAATourneyDetailedResults.csv')
W_seeds    = pd.read_csv(f'{data_dir}/WNCAATourneySeeds.csv')

regular_results = pd.concat([M_regular, W_regular])
tourney_results = pd.concat([M_tourney, W_tourney])
seeds_all       = pd.concat([M_seeds, W_seeds])

season_cutoff = 2003
regular_results = regular_results[regular_results.Season >= season_cutoff]
tourney_results = tourney_results[tourney_results.Season >= season_cutoff]
seeds_all       = seeds_all[seeds_all.Season >= season_cutoff]

print('Raw data loaded.')

Raw data loaded.


In [9]:
def prepare_data(df):
    df = df[["Season", "DayNum", "LTeamID", "LScore", "WTeamID", "WScore", "NumOT",
             "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF",
             "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF"]]
    adjot = (40 + 5 * df["NumOT"]) / 40
    adjcols = [c for c in df.columns if c not in ["Season", "DayNum", "LTeamID", "WTeamID", "NumOT"]]
    for col in adjcols:
        df[col] = df[col] / adjot
    dfswap = df.copy()
    df.columns = [x.replace("W", "T1_").replace("L", "T2_") for x in df.columns]
    dfswap.columns = [x.replace("L", "T1_").replace("W", "T2_") for x in dfswap.columns]
    out = pd.concat([df, dfswap]).reset_index(drop=True)
    out["PointDiff"] = out["T1_Score"] - out["T2_Score"]
    out["win"] = (out["PointDiff"] > 0).astype(int)
    out["men_women"] = out["T1_TeamID"].apply(lambda t: int(str(t).startswith("1")))
    return out

regular_data = prepare_data(regular_results)
tourney_data  = prepare_data(tourney_results)

# Seeds
seeds_all["seed"] = seeds_all["Seed"].apply(lambda x: int(x[1:3]))
seeds_T1 = seeds_all[["Season","TeamID","seed"]].copy().rename(columns={"TeamID":"T1_TeamID","seed":"T1_seed"})
seeds_T2 = seeds_all[["Season","TeamID","seed"]].copy().rename(columns={"TeamID":"T2_TeamID","seed":"T2_seed"})

tourney_data["Total"] = tourney_data["T1_Score"] + tourney_data["T2_Score"]
tourney_data = tourney_data[["Season","T1_TeamID","T2_TeamID","PointDiff","win","men_women","Total"]]
tourney_data = tourney_data.merge(seeds_T1, on=["Season","T1_TeamID"], how="left")
tourney_data = tourney_data.merge(seeds_T2, on=["Season","T2_TeamID"], how="left")
tourney_data["Seed_diff"] = tourney_data["T2_seed"] - tourney_data["T1_seed"]

print('Seeds joined.')

Seeds joined.


In [10]:
# Season averages
boxcols = [
    "T1_Score","T1_FGM","T1_FGA","T1_FGM3","T1_FGA3","T1_FTM","T1_FTA",
    "T1_OR","T1_DR","T1_Ast","T1_TO","T1_Stl","T1_Blk","T1_PF",
    "T2_Score","T2_FGM","T2_FGA","T2_FGM3","T2_FGA3","T2_FTM","T2_FTA",
    "T2_OR","T2_DR","T2_Ast","T2_TO","T2_Stl","T2_Blk","T2_PF","PointDiff",
]
ss = regular_data.groupby(["Season","T1_TeamID"])[boxcols].mean().reset_index()
ss_T1 = ss.copy()
ss_T1.columns = ["T1_avg_" + x.replace("T1_","").replace("T2_","opponent_") for x in ss_T1.columns]
ss_T1 = ss_T1.rename({"T1_avg_Season":"Season","T1_avg_TeamID":"T1_TeamID"}, axis=1)
ss_T2 = ss.copy()
ss_T2.columns = ["T2_avg_" + x.replace("T1_","").replace("T2_","opponent_") for x in ss_T2.columns]
ss_T2 = ss_T2.rename({"T2_avg_Season":"Season","T2_avg_TeamID":"T2_TeamID"}, axis=1)

tourney_data = tourney_data.merge(ss_T1, on=["Season","T1_TeamID"], how="left")
tourney_data = tourney_data.merge(ss_T2, on=["Season","T2_TeamID"], how="left")
print('Season averages joined.')

Season averages joined.


In [11]:
# ELO
base_elo, elo_width, k_factor = 1000, 400, 100

def expected_result(a, b): return 1.0 / (1 + 10**((b - a)/elo_width))
def update_elo(w, l):
    c = k_factor * (1 - expected_result(w, l))
    return w + c, l - c

elos = []
for season in sorted(seeds_all.Season.unique()):
    ss_elo = regular_data[(regular_data.Season == season) & (regular_data.win == 1)].reset_index(drop=True)
    teams = set(ss_elo.T1_TeamID) | set(ss_elo.T2_TeamID)
    elo = dict.fromkeys(teams, base_elo)
    for i in range(len(ss_elo)):
        w, l = ss_elo.loc[i, "T1_TeamID"], ss_elo.loc[i, "T2_TeamID"]
        elo[w], elo[l] = update_elo(elo[w], elo[l])
    elo_df = pd.DataFrame({'TeamID': list(elo.keys()), 'elo': list(elo.values()), 'Season': season})
    elos.append(elo_df)

elos = pd.concat(elos)
elos_T1 = elos.rename(columns={'TeamID':'T1_TeamID','elo':'T1_elo'})
elos_T2 = elos.rename(columns={'TeamID':'T2_TeamID','elo':'T2_elo'})

tourney_data = tourney_data.merge(elos_T1, on=["Season","T1_TeamID"], how="left")
tourney_data = tourney_data.merge(elos_T2, on=["Season","T2_TeamID"], how="left")
tourney_data["elo_diff"] = tourney_data["T1_elo"] - tourney_data["T2_elo"]
print('ELO joined.')

ELO joined.


In [12]:
# GLM quality  (this cell takes a few minutes)
regular_data["ST1"] = regular_data.apply(lambda t: f"{int(t.Season)}/{int(t.T1_TeamID)}", axis=1)
regular_data["ST2"] = regular_data.apply(lambda t: f"{int(t.Season)}/{int(t.T2_TeamID)}", axis=1)
seeds_T1["ST1"] = seeds_T1.apply(lambda t: f"{int(t.Season)}/{int(t.T1_TeamID)}", axis=1)
seeds_T2["ST2"] = seeds_T2.apply(lambda t: f"{int(t.Season)}/{int(t.T2_TeamID)}", axis=1)

st = set(seeds_T1.ST1) | set(seeds_T2.ST2)
st |= set(regular_data.loc[(regular_data.T1_Score > regular_data.T2_Score) & regular_data.ST2.isin(st), "ST1"])

dt = regular_data[regular_data.ST1.isin(st) | regular_data.ST2.isin(st)].copy()
dt["T1_TeamID"] = dt["T1_TeamID"].astype(str)
dt["T2_TeamID"] = dt["T2_TeamID"].astype(str)
dt.loc[~dt.ST1.isin(st), "T1_TeamID"] = "0000"
dt.loc[~dt.ST2.isin(st), "T2_TeamID"] = "0000"

def team_quality(season, mw):
    glm = sm.GLM.from_formula("PointDiff~-1+T1_TeamID+T2_TeamID",
                               data=dt[(dt.Season==season) & (dt.men_women==mw)],
                               family=sm.families.Gaussian()).fit()
    q = pd.DataFrame(glm.params).reset_index()
    q.columns = ["TeamID","quality"]
    q["Season"] = season
    q = q[q.TeamID.str.contains("T1_")].copy()
    q["TeamID"] = q["TeamID"].apply(lambda x: x[10:14]).astype(int)
    return q

glm_quality = []
for s in tqdm.tqdm(sorted(seeds_all.Season.unique()), unit="season"):
    if s >= 2010: glm_quality.append(team_quality(s, 0))
    if s >= 2003: glm_quality.append(team_quality(s, 1))

glm_quality = pd.concat(glm_quality).reset_index(drop=True)
glm_T1 = glm_quality.rename(columns={"TeamID":"T1_TeamID","quality":"T1_quality"})
glm_T2 = glm_quality.rename(columns={"TeamID":"T2_TeamID","quality":"T2_quality"})

tourney_data = tourney_data.merge(glm_T1, on=["Season","T1_TeamID"], how="left")
tourney_data = tourney_data.merge(glm_T2, on=["Season","T2_TeamID"], how="left")
tourney_data["diff_quality"] = tourney_data["T1_quality"] - tourney_data["T2_quality"]
print('GLM quality joined.')

100%|██████████| 23/23 [00:35<00:00,  1.55s/season]

GLM quality joined.


In [13]:
features = [
    "men_women", "T1_seed", "T2_seed", "Seed_diff",
    "T1_avg_Score","T1_avg_FGA","T1_avg_OR","T1_avg_DR","T1_avg_Blk","T1_avg_PF",
    "T1_avg_opponent_FGA","T1_avg_opponent_Blk","T1_avg_opponent_PF","T1_avg_PointDiff",
    "T2_avg_Score","T2_avg_FGA","T2_avg_OR","T2_avg_DR","T2_avg_Blk","T2_avg_PF",
    "T2_avg_opponent_FGA","T2_avg_opponent_Blk","T2_avg_opponent_PF","T2_avg_PointDiff",
    "T1_elo","T2_elo","elo_diff","T1_quality","T2_quality",
]

kaggle_bundle = joblib.load('model/kaggle_proba.joblib')
k_models, spline_model, t = kaggle_bundle['models'], kaggle_bundle['spline'], kaggle_bundle['t']
kaggle_spread = joblib.load('model/kaggle_spread.joblib')
kaggle_total  = joblib.load('model/kaggle_total.joblib')

# OOF win evaluation: use each season's holdout model
test_k = tourney_data[tourney_data.Season.isin(TEST_SEASONS)].copy()

oof_preds, oof_labels = [], []
for season in TEST_SEASONS:
    sub = test_k[test_k.Season == season]
    X_test = xgb.DMatrix(sub[features].values)
    raw_margin = k_models[season].predict(X_test)
    oof_preds  += list(raw_margin)
    oof_labels += list(sub['win'].values)

oof_preds   = np.array(oof_preds)
oof_labels  = np.array(oof_labels)
win_binary  = (oof_preds > 0).astype(int)

k_win_acc     = accuracy_score(oof_labels, win_binary)
proba_calib   = np.clip(spline_model(np.clip(oof_preds, -t, t)), 0.01, 0.99)
k_brier       = brier_score_loss(oof_labels, proba_calib)

# Spread / total (in-sample on 2024-2025)
X_test_df     = test_k[features]
k_spread_mae  = mean_absolute_error(test_k['PointDiff'], kaggle_spread.predict(X_test_df))
k_total_mae   = mean_absolute_error(test_k['Total'],     kaggle_total.predict(X_test_df))

print(f'Kaggle  Win (OOF): {k_win_acc*100:.2f}%   Brier: {k_brier:.5f}')
print(f'Kaggle  Spread MAE (in-sample): {k_spread_mae:.2f} pts')
print(f'Kaggle  Total  MAE (in-sample): {k_total_mae:.2f} pts')

for season in TEST_SEASONS:
    mask = tourney_data.Season == season
    sub_preds  = oof_preds[np.array([tourney_data[tourney_data.Season==s].index[0] <= i <= tourney_data[tourney_data.Season==s].index[-1]
                                     for i, s in zip(range(len(oof_preds)), [season]*len(oof_preds))])]
    sub_labels = test_k[test_k.Season == season]['win'].values
    sub_margin = k_models[season].predict(xgb.DMatrix(test_k[test_k.Season==season][features].values))
    sub_win_acc = accuracy_score(sub_labels, (sub_margin > 0).astype(int))
    print(f'  {season}  Win: {sub_win_acc*100:.2f}%')

rows.append({'Pipeline': 'Kaggle (OOF)', 'Win Accuracy': round(k_win_acc*100,2),
             'Spread MAE': round(k_spread_mae,2), 'Total MAE': round(k_total_mae,2)})

Kaggle  Win (OOF): 80.04%   Brier: 0.13839
Kaggle  Spread MAE (in-sample): 8.26 pts
Kaggle  Total  MAE (in-sample): 10.79 pts
  2024  Win: 77.24%
  2025  Win: 82.84%


---
## Summary

In [14]:
summary = pd.DataFrame(rows)
summary['Win Accuracy'] = summary['Win Accuracy'].apply(lambda x: f'{x:.2f}%')
summary['Spread MAE']   = summary['Spread MAE'].apply(lambda x: f'{x:.2f} pts')
summary['Total MAE']    = summary['Total MAE'].apply(lambda x: f'{x:.2f} pts')
summary['Test Set'] = f'{TEST_SEASONS[0]}–{TEST_SEASONS[-1]}'
summary.loc[summary.Pipeline == 'Kaggle (OOF)', 'Spread MAE'] += ' *'
summary.loc[summary.Pipeline == 'Kaggle (OOF)', 'Total MAE']  += ' *'

print('\n=== MODEL EVALUATION SUMMARY ===')
print(summary.to_string(index=False))
print('\n* Kaggle spread/total trained on all data — these metrics are in-sample for 2024–2025')


=== MODEL EVALUATION SUMMARY ===
    Pipeline Win Accuracy Spread MAE   Total MAE  Test Set
  With Seeds       73.51%  10.07 pts   14.16 pts 2024–2025
    No Seeds       63.06%  11.32 pts   14.16 pts 2024–2025
   Per-Round       71.83%   9.87 pts   14.05 pts 2024–2025
Kaggle (OOF)       80.04% 8.26 pts * 10.79 pts * 2024–2025

* Kaggle spread/total trained on all data — these metrics are in-sample for 2024–2025
